### Imports

In [23]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

import ModelUtils

### Load data from CSV file

In [24]:
# Get data features
data = np.genfromtxt("../data/fma/features.csv", delimiter=',', skip_header=4, dtype=np.float32)

# Get data labels
with open("../data/fma/tracks.csv", encoding="utf-8") as f:
    meta = np.loadtxt(f, delimiter=",", quotechar='"', comments=None,
                      skiprows=3, usecols=(0, 40), dtype=str)
label_ids = meta[:, 0].astype(int)
genres    = meta[:, 1]
has_label = genres != ""
label_ids, genres = label_ids[has_label], genres[has_label]

classes, y_all = np.unique(genres, return_inverse=True)


# Combine into unified varibles
track_ids = data[:, 0].astype(int)
_, feat_rows, label_rows = np.intersect1d(track_ids, label_ids, return_indices=True)

X = data[feat_rows, 1:]
y = y_all[label_rows].astype(np.int64)

### Subdivide training data

In [25]:
perm = np.random.default_rng().permutation(len(X))
split = int(len(X) * 0.8)  # Change 0.8 to change ratio of training to validation data
train_idx, val_idx = perm[:split], perm[split:]

train_ds = torch.utils.data.TensorDataset(torch.from_numpy(X[train_idx]), torch.from_numpy(y[train_idx]))
val_ds   = torch.utils.data.TensorDataset(torch.from_numpy(X[val_idx]),   torch.from_numpy(y[val_idx]))
print(f"Data size: {len(X)}")
print(f"Training size: {len(train_ds)}")
print(f"Validation size: {len(val_ds)}")

Data size: 49598
Training size: 39678
Validation size: 9920


### Define the model

In [26]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using the {device} device for training")

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(X.shape[1], 1027),
            nn.ReLU(),
            nn.Linear(in_features=1027, out_features=594),
            nn.ReLU(),
            nn.Linear(594, 378),
            nn.ReLU(),
            nn.Linear(378, len(classes))
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

Using the cuda device for training


### Train the model

In [27]:
model = NeuralNetwork().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
data_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)

num_epochs = 25

for epoch in range(num_epochs):
    model.train()
    total_loss, total_correct, n = 0.0, 0, 0
    for inputs, targets in data_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = ModelUtils.calculate_function(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(targets)
        total_correct += (outputs.argmax(dim=1) == targets).sum().item()
        n += len(targets)

print(f"Epoch {epoch + 1}: loss {total_loss / n:.4f}, acc {total_correct / n:.3f}")

Epoch 25: loss 1.0992, acc 0.644


### Cache the Model

In [28]:
torch.save(model.state_dict(), "../cache/model.pth")